# 🧹 Notebook 02 – Dataset Preparation

**SpaceSLM-mini** | Dataset Pipeline Step 2

This notebook cleans the raw arXiv data and generates instruction-tuning
QA pairs for fine-tuning.

---

### 🎯 What you will do
1. Load and inspect the raw JSONL data
2. Apply text cleaning (LaTeX removal, unicode normalisation)
3. Filter low-quality and non-space-related abstracts
4. Deduplicate using content hashing
5. Generate instruction / input / output QA triples
6. Split into train / val / eval sets

### 📚 Key concepts
- **Instruction tuning**: Training on (instruction, response) pairs so the model learns to follow commands
- **Alpaca format**: A popular template for instruction datasets: `### Instruction:`, `### Input:`, `### Response:`
- **Data quality**: The model learns from examples — garbage in, garbage out!


In [ ]:
import json, re, hashlib, unicodedata, random
from pathlib import Path
from collections import defaultdict
import pandas as pd

RAW_FILE   = Path('data/raw/arxiv_papers.jsonl')
CLEAN_FILE = Path('data/processed/cleaned_papers.jsonl')
TRAIN_FILE = Path('data/processed/train.jsonl')
VAL_FILE   = Path('data/processed/val.jsonl')
EVAL_FILE  = Path('data/evaluation/eval.jsonl')

assert RAW_FILE.exists(), f'Run notebook 01 first! {RAW_FILE} not found.'
print('✅ Raw data found')

In [ ]:
# Load raw data
raw_papers = []
with open(RAW_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            raw_papers.append(json.loads(line))

print(f'Loaded {len(raw_papers)} raw papers')
df_raw = pd.DataFrame(raw_papers)
df_raw.head(3)

## Step 1: Text Cleaning

arXiv abstracts contain LaTeX math (`$E=mc^2$`), unicode characters, and messy whitespace.  
We clean all of this before generating QA pairs.


In [ ]:
def clean_text(text: str) -> str:
    """Full text cleaning pipeline."""
    # 1. Unicode normalisation
    text = unicodedata.normalize('NFKC', text)
    # 2. Remove display math: $$...$$
    text = re.sub(r'\$\$.*?\$\$', '', text, flags=re.DOTALL)
    # 3. Remove inline math: $...$
    text = re.sub(r'\$[^$]+?\$', '', text)
    # 4. Remove LaTeX commands
    text = re.sub(r'\\[a-zA-Z]+\{[^}]*\}', '', text)
    text = re.sub(r'\\[a-zA-Z]+', '', text)
    text = re.sub(r'[{}]', '', text)
    # 5. Remove non-printable characters
    text = ''.join(ch for ch in text if ch.isprintable())
    # 6. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test it
sample = raw_papers[0]['abstract']
print('BEFORE:', sample[:300])
print('\nAFTER:', clean_text(sample)[:300])

## Step 2: Quality Filtering

In [ ]:
SPACE_KEYWORDS = [
    'satellite', 'spacecraft', 'orbit', 'space', 'lunar', 'mars',
    'asteroid', 'telescope', 'rocket', 'launch', 'mission', 'payload',
    'propulsion', 'solar', 'planetary', 'astronomical', 'astrophysics',
    'cosmology', 'exoplanet', 'nasa', 'esa', 'cubesat',
]

def passes_filter(record: dict) -> tuple[bool, str]:
    abstract = record.get('abstract', '')
    if not abstract:
        return False, 'empty'
    words = abstract.split()
    if len(words) < 30:
        return False, f'too short ({len(words)} words)'
    if len(words) > 600:
        return False, f'too long ({len(words)} words)'
    if not any(kw in abstract.lower() for kw in SPACE_KEYWORDS):
        return False, 'not space-related'
    return True, 'ok'

# Apply filter and show rejection reasons
reasons = defaultdict(int)
for p in raw_papers:
    _, reason = passes_filter({'abstract': clean_text(p.get('abstract', ''))})
    reasons[reason] += 1

print('Filter results:')
for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
    print(f'  {reason:<30} {count:>5}')

## Step 3: Deduplication

In [ ]:
def content_hash(text: str) -> str:
    normalised = ' '.join(text.lower().split())
    return hashlib.sha256(normalised.encode()).hexdigest()

# Full cleaning pipeline
seen_hashes = set()
cleaned_papers = []
stats = {'total': 0, 'passed': 0, 'rejected': 0, 'duplicate': 0}

for paper in raw_papers:
    stats['total'] += 1
    paper['title']    = clean_text(paper.get('title', ''))
    paper['abstract'] = clean_text(paper.get('abstract', ''))
    
    ok, reason = passes_filter(paper)
    if not ok:
        stats['rejected'] += 1
        continue
    
    h = content_hash(paper['abstract'])
    if h in seen_hashes:
        stats['duplicate'] += 1
        continue
    seen_hashes.add(h)
    
    cleaned_papers.append(paper)
    stats['passed'] += 1

# Save cleaned papers
CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(CLEAN_FILE, 'w', encoding='utf-8') as f:
    for p in cleaned_papers:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

print(f'Cleaning stats: {stats}')
print(f'Saved {len(cleaned_papers)} cleaned papers to {CLEAN_FILE}')

## Step 4: Generate Instruction QA Pairs

We convert each abstract into 1–2 instruction-tuning examples using template prompts.
These become the training examples for QLoRA fine-tuning.


In [ ]:
TEMPLATES = [
    {'instruction': 'Explain the main topic of the following research abstract in simple terms.',
     'use_input': True},
    {'instruction': 'Summarise the key findings described in the following research abstract.',
     'use_input': True},
    {'instruction': "What does the research paper titled '{title}' investigate?",
     'use_input': False},
    {'instruction': 'Based on the abstract below, what problem does the research address?',
     'use_input': True},
]

def abstract_to_answer(abstract: str, max_sentences: int = 4) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', abstract.strip())
    return ' '.join(sentences[:max_sentences]).strip()

qa_pairs = []
for paper in cleaned_papers:
    chosen_templates = random.sample(TEMPLATES, k=min(2, len(TEMPLATES)))
    for template in chosen_templates:
        instruction = template['instruction'].format(
            title=paper['title'], abstract=paper['abstract'])
        input_text  = paper['abstract'] if template['use_input'] else ''
        output      = abstract_to_answer(paper['abstract'])
        
        if input_text:
            text = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
        else:
            text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
        
        qa_pairs.append({'instruction': instruction, 'input': input_text,
                         'output': output, 'text': text,
                         'source_id': paper.get('arxiv_id', 'unknown')})

print(f'Generated {len(qa_pairs)} QA pairs from {len(cleaned_papers)} papers')
print('\nSample QA pair:')
print(qa_pairs[0]['text'][:500])

In [ ]:
# Split: 85% train / 10% val / 5% eval
random.seed(42)
random.shuffle(qa_pairs)
n = len(qa_pairs)
n_train = int(n * 0.85)
n_val   = int(n * 0.10)

train_set = qa_pairs[:n_train]
val_set   = qa_pairs[n_train:n_train + n_val]
eval_set  = qa_pairs[n_train + n_val:]

def save_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  Saved {len(records):>5} → {path}')

save_jsonl(train_set, TRAIN_FILE)
save_jsonl(val_set,   VAL_FILE)
save_jsonl(eval_set,  EVAL_FILE)

print(f'\n✅ Dataset ready!')
print(f'   Train: {len(train_set)} | Val: {len(val_set)} | Eval: {len(eval_set)}')

## ✅ Summary

Your training data is ready!

**Next step →** `03_qlora_finetuning.ipynb`  
Open it in Google Colab with a T4 GPU runtime for fine-tuning.
